In [1]:
"""
Detects and removes boilerplate chunks (sidebar/footer/related-posts text
that got scraped as if it were article content) from ayurveda_corpus.json.
 
Detection rule: real article content is essentially never word-for-word
identical across multiple DIFFERENT source articles. A site-wide template
(nav, footer, "related posts" list, category tags) IS identical across many
articles, because it's the same HTML boilerplate repeated on every page.
So: normalize each chunk's text, count how many distinct sources it appears
in verbatim, and drop any chunk whose exact text shows up across more than
`min_sources` different sources - that's a strong, low-false-positive signal
of scraped boilerplate rather than real content.
 
Usage:
    python clean_boilerplate.py
    (reads ayurveda_corpus.json, writes ayurveda_corpus_clean.json in cwd,
     plus a boilerplate_report.json listing exactly what was removed)
"""

'\nDetects and removes boilerplate chunks (sidebar/footer/related-posts text\nthat got scraped as if it were article content) from ayurveda_corpus.json.\n\nDetection rule: real article content is essentially never word-for-word\nidentical across multiple DIFFERENT source articles. A site-wide template\n(nav, footer, "related posts" list, category tags) IS identical across many\narticles, because it\'s the same HTML boilerplate repeated on every page.\nSo: normalize each chunk\'s text, count how many distinct sources it appears\nin verbatim, and drop any chunk whose exact text shows up across more than\n`min_sources` different sources - that\'s a strong, low-false-positive signal\nof scraped boilerplate rather than real content.\n\nUsage:\n    python clean_boilerplate.py\n    (reads ayurveda_corpus.json, writes ayurveda_corpus_clean.json in cwd,\n     plus a boilerplate_report.json listing exactly what was removed)\n'

In [2]:
import json
import os
import re
from collections import defaultdict
 

 
MIN_SOURCES = 3  # text appearing verbatim across at least this many distinct

LINE_OVERLAP_THRESHOLD = 0.5   # fraction of a chunk's lines that must match
                                # known boilerplate lines to flag it
MIN_MATCHING_LINES = 2 
       

In [3]:
"""
Detects and removes boilerplate chunks (sidebar/footer/related-posts text
that got scraped as if it were article content) from ayurveda_corpus.json.

Detection rule: real article content is essentially never word-for-word
identical across multiple DIFFERENT source articles. A site-wide template
(nav, footer, "related posts" list, category tags) IS identical across many
articles, because it's the same HTML boilerplate repeated on every page.
So: normalize each chunk's text, count how many distinct sources it appears
in verbatim, and drop any chunk whose exact text shows up across more than
`min_sources` different sources - that's a strong, low-false-positive signal
of scraped boilerplate rather than real content.

Usage:
    python clean_boilerplate.py
    (reads ayurveda_corpus.json, writes ayurveda_corpus_clean.json in cwd,
     plus a boilerplate_report.json listing exactly what was removed)
"""
import json
import os
import re
from collections import defaultdict

CORPUS_PATH =r"D:\RMIT\SEM 2\Case Studies in Data Science\Ayurvedic_RAG\ayurveda_corpus.json"
CLEAN_CORPUS_PATH =r"D:\RMIT\SEM 2\Case Studies in Data Science\Ayurvedic_RAG\ayurveda_corpus_clean.json"
REPORT_PATH = r"D:\RMIT\SEM 2\Case Studies in Data Science\Ayurvedic_RAG\boilerplate_report.json"

PAIRED_DUPLICATES_REVIEW_PATH = os.path.join(os.getcwd(), 'review_paired_duplicates.json')
STRUCTURAL_REVIEW_PATH = os.path.join(os.getcwd(), 'review_structural_candidates.json')

MIN_SOURCES = 3  # text appearing verbatim across at least this many distinct
                  # sources is treated as boilerplate, not real content

# Second-pass settings: catches boilerplate that got TRUNCATED differently
# across articles (so it's not byte-identical, and Pass 1 above would miss
# it) but still shares most of its individual lines with confirmed
# boilerplate - e.g. a sidebar list cut off at a different point because a
# different article's chunk boundary landed differently.
LINE_OVERLAP_THRESHOLD = 0.5   # fraction of a chunk's lines that must match
                                # known boilerplate lines to flag it
MIN_MATCHING_LINES = 2         # ...and at least this many actual lines,
                                # so a single common short word can't trigger it


def normalize(text):
    """Collapse whitespace differences so near-identical boilerplate (that
    might differ only in blank-line count) still matches as the same text."""
    return re.sub(r'\s+', ' ', text).strip().lower()


def normalize_line(line):
    return re.sub(r'\s+', ' ', line).strip().lower()


def find_boilerplate(corpus, min_sources=MIN_SOURCES):
    text_to_sources = defaultdict(set)
    text_to_chunk_ids = defaultdict(list)

    for chunk in corpus:
        norm = normalize(chunk['content'])
        source = chunk['metadata'].get('source', 'unknown')
        text_to_sources[norm].add(source)
        text_to_chunk_ids[norm].append(chunk['id'])

    boilerplate_texts = {
        norm for norm, sources in text_to_sources.items()
        if len(sources) >= min_sources
    }

    return boilerplate_texts, text_to_sources, text_to_chunk_ids


def extract_known_boilerplate_lines(boilerplate_chunks):
    """
    Pulls individual short lines/phrases out of confirmed (exact-duplicate)
    boilerplate chunks - e.g. 'Categories', 'Arthritis', 'Back Pain'. Even
    when a sidebar gets truncated differently across articles, the
    INDIVIDUAL lines it's built from are usually still identical, since
    it's the same underlying nav/tag list.
    """
    known_lines = set()
    for chunk in boilerplate_chunks:
        for line in chunk['content'].split('\n'):
            norm = normalize_line(line)
            if norm:
                known_lines.add(norm)
    return known_lines


def line_matches_known(norm_line, known_lines, min_shared_words=3):
    """
    True if norm_line exactly matches a known boilerplate line, OR is a
    prefix/truncation of one (or vice versa) - handles boilerplate that got
    cut off mid-sentence at a different point in a different article's
    chunking, so the line itself is shorter than the "reference" version but
    still clearly the same underlying text.
    """
    if norm_line in known_lines:
        return True
    words = norm_line.split()
    if len(words) < min_shared_words:
        return False
    for kl in known_lines:
        if kl.startswith(norm_line) or norm_line.startswith(kl):
            return True
    return False


def find_line_level_boilerplate(remaining_chunks, known_lines,
                                 threshold=LINE_OVERLAP_THRESHOLD,
                                 min_matching=MIN_MATCHING_LINES):
    """
    Second pass: flags chunks that aren't byte-identical to any confirmed
    boilerplate chunk, but are mostly MADE UP of the same known boilerplate
    lines (exact or truncated/prefix matches) - catches partial/truncated
    sidebar variants that exact-duplicate matching (Pass 1) misses.
    """
    flagged = []
    kept = []
    for chunk in remaining_chunks:
        lines = [normalize_line(l) for l in chunk['content'].split('\n') if normalize_line(l)]
        if not lines:
            kept.append(chunk)
            continue
        matching = [l for l in lines if line_matches_known(l, known_lines)]
        overlap_ratio = len(matching) / len(lines)

        if overlap_ratio >= threshold and len(matching) >= min_matching:
            flagged.append((chunk, overlap_ratio, len(matching), len(lines)))
        else:
            kept.append(chunk)

    return kept, flagged


def find_paired_source_duplicates(pass1_survivors, min_words=15):
    """
    Catches content duplicated across exactly 2 sources (Pass 1's MIN_SOURCES=3
    would miss these) - e.g. an article scraped once as itself and once as its
    own comments page, or a podcast episode indexed twice under two different
    feed URL encodings (http vs https). Restricted to chunks with >= min_words,
    since short genuine content (a classical mantra, a common short definition)
    CAN legitimately appear verbatim in two unrelated real sources - only long
    passages repeating verbatim across exactly two sources are treated as safe
    to flag, and even then, this returns a REVIEW list, not an auto-remove.

    Returns (review_list_for_display, matched_chunk_objects) - the chunk
    objects are used elsewhere to extract boilerplate LINES (safe - lines are
    just used to help catch OTHER different chunks, not to auto-delete these
    ones), even though the chunks themselves aren't auto-removed here.
    """
    text_to_sources = defaultdict(set)
    text_to_chunk_ids = defaultdict(list)
    text_to_chunks = defaultdict(list)

    for chunk in pass1_survivors:
        if len(chunk['content'].split()) < min_words:
            continue
        norm = normalize(chunk['content'])
        text_to_sources[norm].add(chunk['metadata'].get('source', 'unknown'))
        text_to_chunk_ids[norm].append(chunk['id'])
        text_to_chunks[norm].append(chunk)

    review = []
    matched_chunks = []
    for norm, sources in text_to_sources.items():
        if len(sources) == 2:
            review.append({
                'preview': norm[:150],
                'sources': sorted(sources),
                'chunk_ids': text_to_chunk_ids[norm],
            })
            matched_chunks.extend(text_to_chunks[norm])
    return review, matched_chunks


def looks_structurally_like_a_list(text, max_line_words=8, min_lines=3, threshold=0.7):
    """
    Flags chunks that read like a navigation list / related-articles sidebar
    even when they only appear ONCE in the whole corpus (so Pass 1/2's
    cross-source duplication logic can't catch them) - e.g. a stray sidebar
    scraped from an unrelated website. Looks for a high proportion of short,
    title-or-question-shaped lines with little actual prose. Report-only -
    single-occurrence content is inherently riskier to auto-remove, since a
    short numbered list COULD be genuine content (e.g. a list of symptoms).
    """
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < min_lines:
        return False
    short_line_count = sum(1 for l in lines if len(l.split()) <= max_line_words)
    return (short_line_count / len(lines)) >= threshold


def find_structural_review_candidates(chunks, threshold=0.7):
    return [
        {'id': c['id'], 'source': c['metadata'].get('source'), 'preview': c['content'][:150].replace('\n', ' ')}
        for c in chunks
        if looks_structurally_like_a_list(c['content'], threshold=threshold)
    ]


def clean_corpus(corpus, min_sources=MIN_SOURCES):
    boilerplate_texts, text_to_sources, text_to_chunk_ids = find_boilerplate(corpus, min_sources)

    pass1_removed = []
    pass1_survivors = []
    for chunk in corpus:
        norm = normalize(chunk['content'])
        if norm in boilerplate_texts:
            pass1_removed.append(chunk)
        else:
            pass1_survivors.append(chunk)

    known_lines = extract_known_boilerplate_lines(pass1_removed)

    # Also learn lines from confirmed 2-source duplicates (review-only chunks
    # aren't auto-removed themselves, but their lines are trustworthy enough
    # to help catch OTHER, different chunks built from the same boilerplate).
    paired_duplicates_review, paired_dup_chunks = find_paired_source_duplicates(pass1_survivors)
    known_lines |= extract_known_boilerplate_lines(paired_dup_chunks)

    # Run Pass 2 only over chunks that AREN'T themselves one of the paired
    # duplicates just detected (those stay in pass1_survivors untouched,
    # visible only via the review file) - Pass 2 targets everything else.
    paired_dup_ids = {c['id'] for c in paired_dup_chunks}
    pass2_candidates = [c for c in pass1_survivors if c['id'] not in paired_dup_ids]
    pass2_untouched = [c for c in pass1_survivors if c['id'] in paired_dup_ids]

    pass2_kept, pass2_flagged = find_line_level_boilerplate(pass2_candidates, known_lines)
    cleaned = pass2_kept + pass2_untouched

    removed = pass1_removed + [f[0] for f in pass2_flagged]

    structural_review = find_structural_review_candidates(cleaned)

    report = []
    for norm in boilerplate_texts:
        report.append({
            'pass': 1,
            'reason': f'exact duplicate across {len(text_to_sources[norm])} sources',
            'preview': norm[:150],
            'appears_in_n_sources': len(text_to_sources[norm]),
            'sources': sorted(text_to_sources[norm])[:10],
            'chunk_ids_removed': text_to_chunk_ids[norm],
            'chunk_count_removed': len(text_to_chunk_ids[norm]),
        })
    for chunk, ratio, matching, total in pass2_flagged:
        report.append({
            'pass': 2,
            'reason': f'{matching}/{total} lines ({ratio:.0%}) matched known boilerplate lines',
            'preview': chunk['content'][:150].replace('\n', ' '),
            'chunk_ids_removed': [chunk['id']],
            'chunk_count_removed': 1,
            'source': chunk['metadata'].get('source'),
        })
    report.sort(key=lambda r: -r['chunk_count_removed'])

    return cleaned, removed, report, paired_duplicates_review, structural_review


def main():
    print(f"Loading corpus from {CORPUS_PATH}...")
    with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
    print(f"Loaded {len(corpus)} chunks.")

    cleaned, removed, report, paired_duplicates_review, structural_review = clean_corpus(corpus)

    print(f"\nFound {len(report)} boilerplate patterns/chunks, "
          f"removing {len(removed)} chunks total ({len(removed)/len(corpus)*100:.1f}% of corpus).")
    pass1_count = sum(r['chunk_count_removed'] for r in report if r.get('pass') == 1)
    pass2_count = sum(r['chunk_count_removed'] for r in report if r.get('pass') == 2)
    print(f"  Pass 1 (exact cross-source duplicates): {pass1_count} chunks")
    print(f"  Pass 2 (partial/truncated line-overlap): {pass2_count} chunks")

    print("\nTop boilerplate patterns found:")
    for r in report[:8]:
        print(f"  - [pass {r['pass']}] {r['reason']}, removed {r['chunk_count_removed']} chunk(s): \"{r['preview']}...\"")

    with open(CLEAN_CORPUS_PATH, 'w', encoding='utf-8') as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    with open(REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    with open(PAIRED_DUPLICATES_REVIEW_PATH, 'w', encoding='utf-8') as f:
        json.dump(paired_duplicates_review, f, ensure_ascii=False, indent=2)
    with open(STRUCTURAL_REVIEW_PATH, 'w', encoding='utf-8') as f:
        json.dump(structural_review, f, ensure_ascii=False, indent=2)

    print(f"\nCleaned corpus saved to: {CLEAN_CORPUS_PATH} ({len(cleaned)} chunks)")
    print(f"Removal report saved to: {REPORT_PATH}")
    print(f"\nNOT auto-removed - for you to review manually:")
    print(f"  {len(paired_duplicates_review)} long passages duplicated across exactly 2 sources -> {PAIRED_DUPLICATES_REVIEW_PATH}")
    print(f"  {len(structural_review)} single-occurrence chunks that look list/sidebar-shaped -> {STRUCTURAL_REVIEW_PATH}")
    print("\nIMPORTANT: you must re-run embed_corpus.py against the CLEAN corpus "
          "to rebuild your embeddings/index - the old embeddings still include "
          "the boilerplate chunks.")


if __name__ == "__main__":
    main()

Loading corpus from D:\RMIT\SEM 2\Case Studies in Data Science\Ayurvedic_RAG\ayurveda_corpus.json...
Loaded 109554 chunks.

Found 59602 boilerplate patterns/chunks, removing 71445 chunks total (65.2% of corpus).
  Pass 1 (exact cross-source duplicates): 14014 chunks
  Pass 2 (partial/truncated line-overlap): 57431 chunks

Top boilerplate patterns found:
  - [pass 1] exact duplicate across 139 sources, removed 278 chunk(s): "contributors jann bellamy, jd scott gavura, bscphm, mba, rph david h. gorski, md, phd harriet hall, md steven p. novella, md kimball c. atwood iv, md ..."
  - [pass 1] exact duplicate across 138 sources, removed 276 chunk(s): "privacy policy sbm translations website terms of use comment policy how to submit a guest post reference links acupuncture chiropractic homeopathy vac..."
  - [pass 1] exact duplicate across 139 sources, removed 139 chunk(s): "recent comments sbm patreon home contact website terms of use science-based medicine exploring issues and controversie

In [6]:
def normalize_line(line):
    return re.sub(r'\s+', ' ', line).strip().lower()
 

In [7]:
def normalize(text):
    """Collapse whitespace differences so near-identical boilerplate (that
    might differ only in blank-line count) still matches as the same text."""
    return re.sub(r'\s+', ' ', text).strip().lower()
 

In [8]:
def find_boilerplate(corpus, min_sources=MIN_SOURCES):
    text_to_sources = defaultdict(set)
    text_to_chunk_ids = defaultdict(list)
 
    for chunk in corpus:
        norm = normalize(chunk['content'])
        source = chunk['metadata'].get('source', 'unknown')
        text_to_sources[norm].add(source)
        text_to_chunk_ids[norm].append(chunk['id'])
 
    boilerplate_texts = {
        norm for norm, sources in text_to_sources.items()
        if len(sources) >= min_sources
    }
 
    return boilerplate_texts, text_to_sources, text_to_chunk_ids
 

In [9]:
def extract_known_boilerplate_lines(boilerplate_chunks):
    """
    Pulls individual short lines/phrases out of confirmed (exact-duplicate)
    boilerplate chunks - e.g. 'Categories', 'Arthritis', 'Back Pain'. Even
    when a sidebar gets truncated differently across articles, the
    INDIVIDUAL lines it's built from are usually still identical, since
    it's the same underlying nav/tag list.
    """
    known_lines = set()
    for chunk in boilerplate_chunks:
        for line in chunk['content'].split('\n'):
            norm = normalize_line(line)
            if norm:
                known_lines.add(norm)
    return known_lines

In [10]:
def find_line_level_boilerplate(remaining_chunks, known_lines,
                                 threshold=LINE_OVERLAP_THRESHOLD,
                                 min_matching=MIN_MATCHING_LINES):
    """
    Second pass: flags chunks that aren't byte-identical to any confirmed
    boilerplate chunk, but are mostly MADE UP of the same known boilerplate
    lines - catches truncated/partially-overlapping sidebar variants that
    exact-duplicate matching (Pass 1) misses.
    """
    flagged = []
    kept = []
    for chunk in remaining_chunks:
        lines = [normalize_line(l) for l in chunk['content'].split('\n') if normalize_line(l)]
        if not lines:
            kept.append(chunk)
            continue
        matching = [l for l in lines if l in known_lines]
        overlap_ratio = len(matching) / len(lines)
 
        if overlap_ratio >= threshold and len(matching) >= min_matching:
            flagged.append((chunk, overlap_ratio, len(matching), len(lines)))
        else:
            kept.append(chunk)
 
    return kept, flagged

In [11]:
def clean_corpus(corpus, min_sources=MIN_SOURCES):
    boilerplate_texts, text_to_sources, text_to_chunk_ids = find_boilerplate(corpus, min_sources)
 
    pass1_removed = []
    pass1_survivors = []
    for chunk in corpus:
        norm = normalize(chunk['content'])
        if norm in boilerplate_texts:
            pass1_removed.append(chunk)
        else:
            pass1_survivors.append(chunk)
 
    known_lines = extract_known_boilerplate_lines(pass1_removed)
    cleaned, pass2_flagged = find_line_level_boilerplate(pass1_survivors, known_lines)
 
    removed = pass1_removed + [f[0] for f in pass2_flagged]
 
    report = []
    for norm in boilerplate_texts:
        report.append({
            'pass': 1,
            'reason': f'exact duplicate across {len(text_to_sources[norm])} sources',
            'preview': norm[:150],
            'appears_in_n_sources': len(text_to_sources[norm]),
            'sources': sorted(text_to_sources[norm])[:10],
            'chunk_ids_removed': text_to_chunk_ids[norm],
            'chunk_count_removed': len(text_to_chunk_ids[norm]),
        })
    for chunk, ratio, matching, total in pass2_flagged:
        report.append({
            'pass': 2,
            'reason': f'{matching}/{total} lines ({ratio:.0%}) matched known boilerplate lines',
            'preview': chunk['content'][:150].replace('\n', ' '),
            'chunk_ids_removed': [chunk['id']],
            'chunk_count_removed': 1,
            'source': chunk['metadata'].get('source'),
        })
    report.sort(key=lambda r: -r['chunk_count_removed'])
 
    return cleaned, removed, report
 

In [12]:
def main():
    print(f"Loading corpus from {CORPUS_PATH}...")
    with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
    print(f"Loaded {len(corpus)} chunks.")
 
    cleaned, removed, report = clean_corpus(corpus)
 
    print(f"\nFound {len(report)} boilerplate patterns/chunks, "
          f"removing {len(removed)} chunks total ({len(removed)/len(corpus)*100:.1f}% of corpus).")
    pass1_count = sum(r['chunk_count_removed'] for r in report if r.get('pass') == 1)
    pass2_count = sum(r['chunk_count_removed'] for r in report if r.get('pass') == 2)
    print(f"  Pass 1 (exact cross-source duplicates): {pass1_count} chunks")
    print(f"  Pass 2 (partial/truncated line-overlap): {pass2_count} chunks")
 
    print("\nTop boilerplate patterns found:")
    for r in report[:8]:
        print(f"  - [pass {r['pass']}] {r['reason']}, removed {r['chunk_count_removed']} chunk(s): \"{r['preview']}...\"")
 
    with open(CLEAN_CORPUS_PATH, 'w', encoding='utf-8') as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    with open(REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
 
    print(f"\nCleaned corpus saved to: {CLEAN_CORPUS_PATH} ({len(cleaned)} chunks)")
    print(f"Removal report saved to: {REPORT_PATH}")
    print("\nIMPORTANT: you must re-run embed_corpus.py against the CLEAN corpus "
          "to rebuild your embeddings/index - the old embeddings still include "
          "the boilerplate chunks.")

In [13]:
if __name__ == "__main__":
    main()

Loading corpus from D:\RMIT\SEM 2\Case Studies in Data Science\Ayurvedic_RAG\ayurveda_corpus.json...
Loaded 109554 chunks.

Found 10222 boilerplate patterns/chunks, removing 22065 chunks total (20.1% of corpus).
  Pass 1 (exact cross-source duplicates): 14014 chunks
  Pass 2 (partial/truncated line-overlap): 8051 chunks

Top boilerplate patterns found:
  - [pass 1] exact duplicate across 139 sources, removed 278 chunk(s): "contributors jann bellamy, jd scott gavura, bscphm, mba, rph david h. gorski, md, phd harriet hall, md steven p. novella, md kimball c. atwood iv, md ..."
  - [pass 1] exact duplicate across 138 sources, removed 276 chunk(s): "privacy policy sbm translations website terms of use comment policy how to submit a guest post reference links acupuncture chiropractic homeopathy vac..."
  - [pass 1] exact duplicate across 139 sources, removed 139 chunk(s): "recent comments sbm patreon home contact website terms of use science-based medicine exploring issues and controversies